# **Exploring the Power of Cross-Validation with the Advertising Dataset - Regression Case**

### Step 1: Load the Dataset

Before diving into the world of model evaluation, let's load our dataset. Our data is saved in a file named `advertising_data.csv`.


In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('advertising_data.csv')

# Display the first few rows to understand its structure
df.head()

,TV,Radio,Newspaper,Sales
0,115.489335,32.101582,10.312387,25.314828
1,285.460720,4.206998,90.255291,35.364573
2,220.938213,8.081436,50.525237,30.434573
3,181.604253,44.927709,82.645747,38.310782
4,51.025499,30.321453,32.004960,17.230792


## Step 2: A Quick Recap on Cross-Validation

Cross-validation is a technique used to assess the performance of machine learning models more robustly than a simple train-test split. It works by dividing the dataset into multiple subsets or "folds". The model is trained on some of these folds and tested on the remaining fold. This process is repeated multiple times, with each fold getting a turn as the test set.

**Advantages:**

- **More Robust Evaluation:** By training and testing on different data splits, we get a better idea of a model's overall performance.

- **Better Utilization of Data:** Since every data point gets to be in both the training and test sets, we're making efficient use of available data.

## Step 3: Preparing the Data

We'll be predicting Sales using TV, Radio, and Newspaper as predictors.


In [2]:
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

In [3]:
from sklearn.model_selection import train_test_split
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Step 4: Implementing Cross-Validation

We'll use the LinearRegression model and perform 5-Fold Cross-Validation.


In [4]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

# Initialize the model
model = LinearRegression()

## Metrics for Single Train-Test Split

After training a model on a training dataset and making predictions on a test dataset, we can compute the following metrics:

- **Mean Absolute Error (MAE):** This metric calculates the average absolute differences between the actual and predicted values. It gives you an idea of how far off your predictions are on average.

- **Mean Squared Error (MSE):** MSE measures the average of the squared differences between actual and predicted values. It penalizes larger errors more heavily, making it sensitive to outliers.

- **Root Mean Squared Error (RMSE):** RMSE is the square root of the MSE. It provides a metric in the same unit as the target variable, which can be easier to interpret.

- **R-squared (R2) Score:** R2 measures the proportion of the variance in the target variable that is predictable from the independent variables. It ranges from 0 to 1, with higher values indicating a better fit of the model to the data.

These metrics help assess the performance and accuracy of your model when applied to a single train-test split of the data.


In [5]:
# Fit the mode
model.fit(X_train, y_train)

LinearRegression()

In [6]:
y_pred = model.predict(X_test)

In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Metrics for Single Train-Test Split

mse_single = mean_squared_error(y_test, y_pred)
rmse_single = np.sqrt(mse_single)
mad_single = mean_absolute_error(y_test, y_pred)
r2_single = r2_score(y_test, y_pred)

In [8]:
print(f"Metrics with Single Train-Test Split:")
print(f"MSE: {mse_single:.2f}")
print(f"RMSE: {rmse_single:.2f}")
print(f"MAD: {mad_single:.2f}")
print(f"R^2: {r2_single:.2f}")

Metrics with Single Train-Test Split:
MSE: 6.21
RMSE: 2.49
MAD: 2.11
R^2: 0.96


# Understanding cross_validate for Regression Case

The `cross_validate` function is a powerful tool in the scikit-learn library that allows us to evaluate the performance of machine learning models using cross-validation. It offers more flexibility than `cross_val_score` by providing detailed information about various evaluation metrics and optionally returning additional information about fit times and score times.

It returns a dict containing fit-times, score-times (and **optionally training scores** as well as fitted estimators) in addition to the test score.

**Parameters:**

- **estimator:** This is the machine learning model that you want to evaluate. It should be an instance of a model that implements the "fit" method (e.g., `LinearRegression()`).

- **X:** The predictor or feature matrix. It is the data that you want to use to predict the target variable.

- **y:** The target variable or what you're trying to predict.

- **cv:** The cross-validation splitting strategy. It can be a number, an instance of cross-validation generator, or even a TimeSeriesSplit object for time series data. Common values for `cv` are 5 or 10.

- **scoring:** This parameter determines the metric(s) to be used for evaluation. The parameter can accept a single string or a list of strings representing desired evaluation metrics. Common metrics include `'neg_mean_squared_error'`, `'r2'`, `'accuracy'`, etc.

  - The `"neg"` in `'neg_mean_squared_error'` implies that negative MSE values are returned because, in general, scoring functions in scikit-learn follow the convention that higher values are better. So, by taking the negative of MSE (which we usually want to minimize), it aligns with this convention.

**Usage:**

```python
from sklearn.model_selection import cross_validate
scoring = ['r2', 'neg_mean_squared_error']
results = cross_validate(estimator, X, y, cv=5, scoring=scoring)
```

In [9]:
from sklearn.model_selection import cross_validate
import numpy as np

# Define the estimator
estimator = LinearRegression()

# Define the scoring metrics
scoring = {
    'neg_mean_squared_error': 'neg_mean_squared_error',
    'neg_mean_absolute_error': 'neg_mean_absolute_error',
    'r2': 'r2'
}

# Suffle the data
from sklearn.utils import shuffle
X_shuffle, y_shuffle = shuffle(X, y)

# Perform cross-validation
cv_results = cross_validate(estimator, X_shuffle, y_shuffle, cv=5, scoring=scoring)

# Calculate the metrics
mse_cv = -cv_results['test_neg_mean_squared_error'].mean()
rmse_cv = np.sqrt(mse_cv)
mad_cv = -cv_results['test_neg_mean_absolute_error'].mean()
r2_cv = cv_results['test_r2'].mean()

In [10]:
print("\nMetrics with 5-Fold Cross Validation:")
print(f"MSE: {mse_cv:.2f}")
print(f"RMSE: {rmse_cv:.2f}")
print(f"MAD: {mad_cv:.2f}")
print(f"R^2: {r2_cv:.2f}")


Metrics with 5-Fold Cross Validation:
MSE: 9.66
RMSE: 3.11
MAD: 2.48
R^2: 0.93


# Model Performance: Single Train-Test Split vs. Cross Validation

| **Metric** | **Single Train-Test Split** | **Cross Validation** |
| ------ | ----------------------- | ----------------- |
| **MSE**    | **6.21**                    | **9.54**              |
| **RMSE**   | **2.49**                    | **3.09**              |
| **MAD**    | **2.11**                    | **2.45**              |
| **R^2**    | **0.96**                    | **0.93**              |

**Explanation:**

The table above lists the performance metrics of a model using a **single train-test split** and using **cross-validation**.

**Discussion:**

From the above table, we observe that the metrics from the **single train-test split** are slightly better than those from **cross-validation**. While it might be tempting to consider the **single split method superior** due to these results, it's essential to understand the benefits of **cross-validation**.

**Cross-validation** provides a more comprehensive evaluation by training and testing on different subsets of the data. This process can highlight issues like **overfitting** that might not be apparent with a **single split**.

In our case, the slight degradation in metrics with **cross-validation** might suggest that while our model performs well on the specific **single split**, its performance can vary with different data subsets. Hence, using **cross-validation** provides a more generalized and robust estimate of the model's performance on unseen data.


# Assessing for Overfitting


- If the model performs significantly better on the training data compared to the validation data (higher training scores), it may be overfitting.



In [11]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
import numpy as np

# Create your dataset and target variable (X, y)

# Define your model
estimator = LinearRegression()  # Replace with your chosen estimator

# Define evaluation metrics (e.g., 'neg_mean_squared_error' for MSE)
scoring_metrics = ['neg_mean_squared_error', 'r2']

# Perform cross-validation with 'return_train_score=True' to include training scores
cv_results = cross_validate(estimator, X, y, cv=5, scoring=scoring_metrics, return_train_score=True)

In [12]:
cv_df = pd.DataFrame(cv_results)

In [13]:
cv_df

,fit_time,score_time,test_neg_mean_squared_error,train_neg_mean_squared_error,test_r2,train_r2
0,0.003293,0.002795,-11.124169,-8.739317,0.923834,0.938603
1,0.002500,0.002410,-8.000354,-9.484110,0.944888,0.933575
2,0.002412,0.002374,-8.110780,-9.461073,0.949845,0.932089
3,0.003235,0.002974,-9.146135,-9.192577,0.924652,0.938514
4,0.002445,0.002333,-11.106217,-8.715304,0.918020,0.940009


In [14]:
# Extract training and testing scores for each metric
train_mse = -cv_results['train_neg_mean_squared_error']
test_mse = -cv_results['test_neg_mean_squared_error']
train_r2 = cv_results['train_r2']
test_r2 = cv_results['test_r2']

# Calculate the mean training and testing scores for each metric
mean_train_mse = np.mean(train_mse)
mean_test_mse = np.mean(test_mse)
mean_train_r2 = np.mean(train_r2)
mean_test_r2 = np.mean(test_r2)

# Print the results for comparing training vs testing
print("Comparing Training vs Testing in Cross-Validation:")
print(f"Mean Training MSE: {mean_train_mse:.2f}")
print(f"Mean Testing MSE: {mean_test_mse:.2f}")
print(f"Mean Training R^2: {mean_train_r2:.2f}")
print(f"Mean Testing R^2: {mean_test_r2:.2f}")

Comparing Training vs Testing in Cross-Validation:
Mean Training MSE: 9.12
Mean Testing MSE: 9.50
Mean Training R^2: 0.94
Mean Testing R^2: 0.93
